# 🤖 Production-Grade Stateful AI Agent with Persistent Memory & Dynamic Summarization

**Author / Developer:** AI Engineer Portfolio Project  
**Category:** Generative AI & Agentic Systems  
**Tech Stack:** Python 3.14 | LangGraph | LangChain Core | Google Gemini (`ChatGoogleGenerativeAI`) | SQLite (`SqliteSaver`) | Pydantic  

---

## 📌 Executive Architecture & Engineering Overview
In this project, I engineered a production-grade, stateful AI agent architecture utilizing **LangGraph**, **LangChain**, **Google Gemini**, and **SQLite**.

Traditional LLM chains are inherently stateless, requiring the full conversation history to be resent with every API call. This leads to quadratic token growth, soaring API costs, and context window exhaustion. To solve this, I designed and implemented an autonomous, cyclic state-machine architecture that combines:

1. **Typed State Schema with Reducer Semantics (`add_messages`):** Built upon `MessagesState` to guarantee deterministic state updates, safe message appending, and selective message deletion.
2. **Autonomous Context Condensation & Token Optimization:** Dynamic LLM-driven conversation distillation that compresses historical turns into a cumulative summary and evicts raw messages with `RemoveMessage` when token thresholds are exceeded.
3. **Specialized Multi-Node Graph Pipeline:** Discrete functional nodes separating user input ingestion, context-augmented LLM reasoning, continuation prompting, and conversation distillation.
4. **Dynamic Conditional Routing:** Intelligent decision edges that inspect state volume and user intent to steer execution flow cyclically or terminate gracefully.
5. **ACID-Compliant Multi-Thread Checkpointing (`SqliteSaver`):** Thread-isolated session management (`thread_id`) enabling multi-user concurrency and fault-tolerant state recovery.
6. **Time-Travel Trajectory Auditing:** Deep observability tools via `get_state_history()` to inspect, debug, and audit state transitions at each checkpoint.


---
## 1. Environment Configuration & Library Imports

In this section, I configure the environment and import core primitives from **LangGraph** (state graph construction, state schemas, message reducers, and SQLite checkpointers) alongside **LangChain Core** (message schemas) and **LangChain Google GenAI** (`ChatGoogleGenerativeAI`).


In [ ]:
%load_ext dotenv
%dotenv
%load_ext mypy_ipython

import os
import sqlite3
from typing import Literal
from typing_extensions import TypedDict
from collections.abc import Sequence

# LangChain message primitives
from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    AIMessage,
    SystemMessage,
    RemoveMessage
)

# LangGraph core primitives
from langgraph.graph import (
    START,
    END,
    StateGraph,
    MessagesState,
    add_messages
)

# Memory and Checkpointing
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver

# Language Model (Google Gemini)
from langchain_google_genai import ChatGoogleGenerativeAI


---
## 2. Comprehensive State Schema Definition

Here, I designed a strongly-typed `AgentState` schema inheriting from LangGraph's `MessagesState`.

By extending `MessagesState`, the `messages` list automatically leverages the `add_messages` reducer to safely append incoming dialogue and process deletions via `RemoveMessage`. I introduced an additional `summary: str` field to persist high-level conversational memory across conversation cycles without bloating the prompt window.


In [ ]:
class AgentState(MessagesState):
    """
    Comprehensive Agent State:
    - messages: Sequence[BaseMessage] managed by add_messages reducer
    - summary: str containing LLM-condensed context of earlier conversation
    """
    summary: str


---
## 3. LLM Model Initialization

I initialized Google Gemini using `ChatGoogleGenerativeAI` (configurable via `GEMINI_MODEL`, defaulting to `gemini-2.5-flash`) with a balanced temperature setting (`0.7`) to deliver fluent, context-aware responses and reliable dialogue summarization.


In [ ]:
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
gemini_model = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")

chat = ChatGoogleGenerativeAI(
    model=gemini_model,
    google_api_key=gemini_api_key,
    temperature=0.7
)


---
## 4. Agent Node Implementations

I implemented 4 specialized, decoupled nodes to handle each phase of the agent's lifecycle:
1. `ask_question`: Ingests the user's initial inquiry and records dialogue into state as typed messages (`AIMessage` and `HumanMessage`).
2. `chatbot`: Injects any existing cumulative summary as a `SystemMessage` context anchor, appends active message history, and queries Google Gemini.
3. `ask_another_question`: Gathers continuation intent from the user for cyclical multi-turn dialogue.
4. `summarize_messages`: An autonomous context optimizer that invokes Gemini to condense dialogue history, updates the cumulative `summary`, and emits `RemoveMessage` tokens to prune historical messages from active memory.


In [ ]:
def ask_question(state: AgentState) -> AgentState:
    """Node 1: Collects user question and logs state."""
    print("\n======> [NODE: ask_question]")
    question = "What is your question?"
    print(f"Assistant: {question}")
    
    user_input = input("You: ")
    return AgentState(messages=[AIMessage(question), HumanMessage(user_input)])


In [ ]:
def chatbot(state: AgentState) -> AgentState:
    """Node 2: Generates assistant response, incorporating cumulative summary if present."""
    print("\n======> [NODE: chatbot]")
    
    current_summary = state.get("summary", "")
    messages_to_send = []
    
    if current_summary:
        system_instruction = f'''
Here is a concise summary of the conversation so far:
{current_summary}

Keep this context in mind when answering the following user message.
'''
        messages_to_send.append(SystemMessage(system_instruction))
    
    messages_to_send.extend(state["messages"])
    response = chat.invoke(messages_to_send)
    
    assistant_text = getattr(response, "text", "") or (response.content if isinstance(response.content, str) else str(response.content))
    print(f"Assistant: {assistant_text}")
    return AgentState(messages=[response])


In [ ]:
def ask_another_question(state: AgentState) -> AgentState:
    """Node 3: Prompts user whether to continue."""
    print("\n======> [NODE: ask_another_question]")
    prompt = "Would you like to ask another question (yes/no)?"
    print(f"Assistant: {prompt}")
    
    user_choice = input("You: ")
    return AgentState(messages=[AIMessage(prompt), HumanMessage(user_choice)])


In [ ]:
def summarize_messages(state: AgentState) -> AgentState:
    """Node 4: Compresses conversation history into cumulative summary and prunes messages."""
    print("\n======> [NODE: summarize_messages]")
    
    # Format active conversation history
    conversation_text = ""
    for msg in state["messages"]:
        msg_text = getattr(msg, "text", "") or (msg.content if isinstance(msg.content, str) else str(msg.content))
        conversation_text += f"{msg.type.upper()}: {msg_text}\n\n"
    
    summary_prompt = f'''
Update the ongoing conversation summary by incorporating the new lines of dialogue below.
Build upon the previous summary rather than repeating it.
Respond ONLY with the updated summary text.

Previous Summary:
{state.get("summary", "")}

New Dialogue:
{conversation_text}
'''
    updated_summary = chat.invoke([HumanMessage(summary_prompt)])
    summary_text = getattr(updated_summary, "text", "") or (updated_summary.content if isinstance(updated_summary.content, str) else str(updated_summary.content))
    print(f"[SUMMARY UPDATED]: {summary_text[:120]}...")
    
    # Evict messages that have been summarized using RemoveMessage
    remove_tokens = [RemoveMessage(id=msg.id) for msg in state["messages"] if hasattr(msg, "id") and msg.id]
    
    return AgentState(
        messages=remove_tokens,
        summary=summary_text
    )


---
## 5. Dynamic Routing Function & Conditional Edges

To enable autonomous decision-making, I engineered a conditional routing function `routing_function`:
- **Context Threshold Check:** If the active message list exceeds **4 messages**, it routes to `summarize_messages` to distill context and protect against token bloat.
- **Intent Evaluation:** Otherwise, it parses user intent. If affirmative (`yes`, `y`, `sure`), it loops cyclically back to `ask_question`; if negative or complete, it routes to `__end__`.


In [ ]:
def routing_function(state: AgentState) -> Literal["ask_question", "summarize_messages", "__end__"]:
    """Inspects state to determine next graph transition."""
    
    # 1. Trigger context condensation if message history grows
    if len(state["messages"]) > 4:
        return "summarize_messages"
    
    # 2. Check user continuation intent
    latest_msg = state["messages"][-1]
    latest_text = getattr(latest_msg, "text", "") or (latest_msg.content if isinstance(latest_msg.content, str) else str(latest_msg.content))
    latest_user_message = latest_text.strip().lower()
    
    if latest_user_message in ["yes", "y", "sure", "ok"]:
        return "ask_question"
    else:
        return "__end__"


---
## 6. Graph Construction & Persistent Checkpointing

In this step, I assembled the `StateGraph` topology, wired sequential edges, registered conditional routing edges, and compiled the graph with an ACID-compliant SQLite checkpointer (`SqliteSaver`).

This persistent checkpointer ensures that conversations survive process interruptions, scale across multi-tenant sessions, and maintain state integrity across unique `thread_id` identifiers.


In [ ]:
# Initialize StateGraph
graph = StateGraph(AgentState)

# Add Nodes
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)
graph.add_node("summarize_messages", summarize_messages)

# Connect Sequential Edges
graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_edge("summarize_messages", "ask_question")

# Add Conditional Routing Edges
graph.add_conditional_edges(
    source="ask_another_question",
    path=routing_function
)

# Connect Persistent Checkpointer (SQLite)
db_connection = sqlite3.connect("langgraph_memory.db", check_same_thread=False)
checkpointer = SqliteSaver(db_connection)

# Compile Graph
agent_app = graph.compile(checkpointer=checkpointer)
print("StateGraph compiled successfully with SQLite checkpointing!")


---
## 7. Graph Topology Visualization

To verify graph integrity and edge transitions, I rendered the compiled execution topology in ASCII representation.


In [ ]:
print(agent_app.get_graph().draw_ascii())


---
## 8. Multi-Thread Execution & Thread Isolation

Here, I executed the agent under an isolated session configuration (`thread_id="session_001"`).

LangGraph isolates conversation states per thread, ensuring that multiple independent user sessions can run concurrently against the same SQLite database without state contamination.


In [ ]:
config_thread_1 = {"configurable": {"thread_id": "session_001"}}

# Execute workflow
# (Will step through ask_question -> chatbot -> ask_another_question -> conditional route)
output_state = agent_app.invoke(AgentState(messages=[], summary=""), config=config_thread_1)


---
## 9. State History Inspection & Time-Travel Auditing

One of the most powerful features I incorporated is state observability. By leveraging `get_state_history()`, we can inspect the full chronological trajectory of checkpoints, examine intermediate state snapshots, and perform time-travel debugging across the agent's entire execution lifecycle.


In [ ]:
# Retrieve all historical snapshots for this thread
history = list(agent_app.get_state_history(config_thread_1))
print(f"Total historical checkpoints recorded for session_001: {len(history)}\n")

for i, snapshot in enumerate(history[::-1]):
    print(f"Checkpoint {i+1}: Node '{snapshot.next}' | Values: {list(snapshot.values.keys())}")
    if snapshot.values.get("summary"):
        print(f"   -> Summary: {snapshot.values['summary'][:80]}...")


In [ ]:
%mypy
